In [ ]:
!pip uninstall -y transformers trl peft bitsandbytes accelerate torchao
!pip install -q \
transformers==4.53.2 \
trl==0.19.1 \
peft==0.16.0 \
accelerate==1.8.1 \
bitsandbytes==0.46.1 \
huggingface_hub==0.34.4 \
datasets==4.0.0

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_name = "google/gemma-2-2b-it"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", attn_implementation="eager")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.01,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type = TaskType.CAUSAL_LM
)

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()

model = get_peft_model(model, lora_config)

In [ ]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:10000]")

In [ ]:
print(dataset)
print(dataset.shape)
print(len(dataset))

In [ ]:
def format_df(example):
  return{
      "text" : tokenizer.apply_chat_template(
          example["messages"], tokenize=False, add_generation_prompt=False
      )
  }

dataset = dataset.map(format_df, remove_columns=dataset.column_names)

In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(len(train_dataset))
print(len(test_dataset))

In [ ]:
args = SFTConfig(
    output_dir = "./gemma2-sft",
    num_train_epochs = 1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    save_strategy = "epoch",
    eval_strategy = "epoch",
    learning_rate = 2e-5,
    fp16 = True,
    bf16 = False,
    gradient_checkpointing = True,
    max_length = 1024,
    logging_steps=10,
    logging_strategy="steps",
    report_to="none",
    push_to_hub=True,
    hub_model_id="ciphermosaic/gemma-2-2b-sft",
    hub_strategy="every_save" # or "end"
)

In [ ]:
# solves bfloat16 problem.
for name, p in model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

for name, p in model.named_parameters():
    if p.requires_grad:
      assert p.dtype == torch.float32, f"{name} is {p.dtype}"
print("ALL are float32")

In [ ]:
# from huggingface_hub import whoami
# print(whoami())

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    args = args
    )

# Training

In [ ]:
# !nvidia-smi

In [ ]:
model.print_trainable_parameters()

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./gemma2-sft")
trainer.push_to_hub("ciphermosaic/gemma-2-2b-sft")

# Inference

In [ ]:
from transformers import pipeline

base_pipe = pipeline("text-generation", model=model_name, device_map="auto")
tuned_pipe = pipeline("text-generation", model="gemma-2-2b-sft", device_map="auto")

test_prompts = [
    "what should i gift my mom on her birthday?",
    "Explain quantum computing simple."
]

for p in test_prompts:
  chat = [{"role" : "user", "content" : p}]
  formatted = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompts=True)

  print("BASE: ", base_pipe(formatted, max_new_tokens=200)[0]["generated-text"])
  print("TUNED: ", tuned_pipe(formatted, max_new_tokens=200)[0]["generated-text"])

save merge

In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("gemma-2-2b-sft-metged", safe_serialization=True)

tokenizer.save_pretrained("gemma-2-2b-sft-merged")

push merge

In [ ]:
merged_model.push_to_hub("ciphermosaic/gemma-2-2b-sft-merged")
tokenizer.push_to_hub("ciphermosaic/gemma-2-2b-sft-merged")

install llama.cpp

In [ ]:
!git clone https://github.com/ggml-org/llama.Cpp
!pip install -r llama.cpp/requirements.txt

# convert to gguf

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py \
    gemma2-2b-sft-merged \
    --outfile gemma2-2b-sft-Q4_K_M.gguf \
    --outtype q4_k_m

In [ ]:
from google.colab import files
files.download("gemma2-2b-sft-Q4_K_M.gguf")

In [ ]:
!huggingface-cli upload ciphermosaic/gemma2-2b-sft-gguf \
    gemma2-2b-sft-Q4_K_M.gguf